In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import os
import numpy as np
import matplotlib.pyplot as plt
from torch.amp import GradScaler, autocast


# для colab
# import sys
# sys.path.append('/content/drive/MyDrive')

from model import SASRecModel

# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
def ndcg_at_k(rel, pred, k=10):
    ndcg = 0.0
    if rel in pred[:k]:
        pred_list = list(pred[:k])
        score = pred_list.index(rel) + 1
        ndcg = 1.0 / np.log2(score + 1)
        return ndcg
    return ndcg


def recall_at_k(rel, pred, k=10):
    recall = 1.0 if rel in pred[:k] else 0.0
    return recall

In [ ]:
def info_nce_loss(item_emb, author_ids, temp=0.1):
    # сближает эмбеддинги книг одного автора
    batch_size = item_emb.shape[0]
    item_emb = F.normalize(item_emb, dim=1)
    sim = torch.mm(item_emb, item_emb.t()) / temp
    author_ids = author_ids.unsqueeze(0)
    mask = (author_ids == author_ids.t()).float()
    mask = mask - torch.eye(batch_size, device=mask.device)
    exp_sim = torch.exp(sim)
    pos_sum = (exp_sim * mask).sum(dim=1)
    sum_all = exp_sim.sum(dim=1) - torch.exp(torch.diag(sim))
    loss = -torch.log(pos_sum / sum_all + 1e-8)
    sum_mask = mask.sum(dim=1)
    loss = (loss * (sum_mask > 0).float()).sum() / (sum_mask > 0).float().sum() if (sum_mask > 0).any() else torch.tensor(0.0, device=loss.device)
    return loss

In [ ]:
def item_dropout(inputs, authors, categories, dropout_rate=0.15):
    # случайно заменяет 15% книг на padding
    mask = torch.rand_like(inputs.float()) > dropout_rate
    mask[inputs == 0] = True
    dropped_inputs = inputs.clone()
    dropped_inputs[~mask] = 0
    dropped_authors = authors.clone()
    dropped_authors[~mask] = 0
    dropped_categories = categories.clone()
    dropped_categories[~mask] = 0
    return dropped_inputs, dropped_authors, dropped_categories

In [ ]:
# для colab
# data = torch.load('/content/drive/MyDrive/preprocessed_data.pt')

data = torch.load('preprocessed_data.pt')
train_inputs = data['train_inputs']
train_targets = data['train_targets']
train_authors = data['train_authors']
train_categories = data['train_categories']
validate_inputs = data['validate_inputs']
validate_targets = data['validate_targets']
validate_authors = data['validate_authors']
validate_categories = data['validate_categories']
cnt_item = data['cnt_item']
cnt_author = data['cnt_author']
cnt_category = data['cnt_category']

print(f"Train: {len(train_inputs)} примеров")
print(f"Validate: {len(validate_inputs)} примеров")

In [ ]:
batch_size = 32
train_dataset = TensorDataset(train_inputs, train_targets,
                               train_authors, train_categories)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = SASRecModel(cnt_item=cnt_item, max_seq_len=30,
                     hidden_dim=64, num_heads=2, num_layers=2, dropout=0.2,
                     cnt_authors=cnt_author, cnt_categories=cnt_category).to(device)
print(f"Параметров: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def check_metrics_validate(model, validate_inputs, validate_targets,
                           validate_authors, validate_categories,
                           k=10, batch_size=16):
    model.eval()
    ndcg_scores, recall_scores = [], []

    with torch.no_grad():
        for i in tqdm(range(0, len(validate_inputs), batch_size), desc="Validate"):
            batch_input = validate_inputs[i:i+batch_size].to(device)
            batch_target = validate_targets[i:i+batch_size]
            batch_authors = validate_authors[i:i+batch_size].to(device)
            batch_categories = validate_categories[i:i+batch_size].to(device)

            logits = model(batch_input, batch_authors, batch_categories)
            scores = logits[:, -1, :]
            _, pred = torch.topk(scores, k=k, dim=1)
            pred = pred.cpu().numpy()

            for j in range(len(batch_target)):
                target = batch_target[j].item()
                ndcg_scores.append(ndcg_at_k(target, pred[j], k))
                recall_scores.append(recall_at_k(target, pred[j], k))

    model.train()
    return np.mean(ndcg_scores), np.mean(recall_scores)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scaler = GradScaler('cuda')
os.makedirs('checkpoints', exist_ok=True)

best_ndcg = 0.0
check_validate = 2
patience = 2
patience_counter = 0
alpha = 0.1
dropout_rate = 0.15

history = {
    'epoch': [],
    'train_loss': [],
    'validate_ndcg': [],
    'validate_recall': []
}

print(f"\nContrastive Learning + Item Dropout (drop={dropout_rate})")

for epoch in range(1, 25):
    model.train()
    total_loss = 0

    for inputs, targets, authors, categories in tqdm(train_loader, desc=f"Epoch {epoch}/25"):
        inputs, targets = inputs.to(device), targets.to(device)
        authors, categories = authors.to(device), categories.to(device)

        # Item Dropout
        dropped_inputs, dropped_authors, dropped_categories = item_dropout(
            inputs, authors, categories, dropout_rate
        )

        with autocast('cuda'):
            logits = model(dropped_inputs, dropped_authors, dropped_categories)
            rec_loss = negative_sampling_loss(logits[:, -1, :], targets, cnt_item)
            target_emb = model.item_emb(targets)
            target_authors = authors[:, -1]
            contrastive_loss = info_nce_loss(target_emb, target_authors)
            loss = rec_loss + alpha * contrastive_loss

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    history['epoch'].append(epoch)
    history['train_loss'].append(avg_loss)

    if epoch % check_validate == 0:
        validate_ndcg, validate_recall = check_metrics_validate(
            model, validate_inputs, validate_targets,
            validate_authors, validate_categories, k=10, batch_size=16
        )
        history['validate_ndcg'].append(validate_ndcg)
        history['validate_recall'].append(validate_recall)

        print(f'Epoch {epoch:2d} | Train Loss: {avg_loss:.4f} | Validate NDCG@10: {validate_ndcg:.4f} | Validate Recall@10: {validate_recall:.4f}')

        if validate_ndcg > best_ndcg:
            best_ndcg = validate_ndcg
            patience_counter = 0
            torch.save(model.state_dict(), 'checkpoints/contrastive_best.pth')
            print(f'Лучшая модель сохранена (NDCG@10: {best_ndcg:.4f})')
        else:
            patience_counter += 1
            print(f'Нет улучшения ({patience_counter}/{patience})')

        if patience_counter >= patience:
            print(f'Ранняя остановка на эпохе {epoch}')
            break
    else:
        history['validate_ndcg'].append(None)
        history['validate_recall'].append(None)
        print(f'Epoch {epoch:2d} | Train Loss: {avg_loss:.4f}')

print(f'Лучшая метрика NDCG@10 на валидации: {best_ndcg:.4f}')

In [ ]:
plt.figure()
plt.plot(history['epoch'], history['train_loss'], marker='o')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.title('Train Loss')
plt.grid(True)
plt.show()

val_epochs = [e for e, v in zip(history['epoch'], history['validate_ndcg']) if v is not None]
val_ndcg_clean = [v for v in history['validate_ndcg'] if v is not None]
val_recall_clean = [v for v in history['validate_recall'] if v is not None]

plt.figure()
plt.plot(val_epochs, val_ndcg_clean, marker='s', label='NDCG@10')
plt.plot(val_epochs, val_recall_clean, marker='^', label='Recall@10')
plt.xlabel('Эпоха')
plt.ylabel('Метрика')
plt.title('Качество на валидации')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
test_inputs = data['test_inputs']
test_targets = data['test_targets']
test_authors = data['test_authors']
test_categories = data['test_categories']
k_values = [10, 50, 100]

model.load_state_dict(torch.load('checkpoints/contrastive_best.pth', map_location=device))
model.eval()

ndcg_scores = {k: [] for k in k_values}
recall_scores = {k: [] for k in k_values}

with torch.no_grad():
    for i in tqdm(range(len(test_inputs)), desc="Тест"):
        inp = test_inputs[i].unsqueeze(0).to(device)
        auth = test_authors[i].unsqueeze(0).to(device)
        cat = test_categories[i].unsqueeze(0).to(device)
        target = test_targets[i].item()
        scores = model(inp, auth, cat)[0, -1, :]
        _, pred = torch.topk(scores, k=max(k_values), dim=0)
        pred = pred.cpu().numpy()
        for k in k_values:
            ndcg_scores[k].append(ndcg_at_k(target, pred, k))
            recall_scores[k].append(recall_at_k(target, pred, k))

for k in k_values:
    print(f"NDCG@{k:<3}   {np.mean(ndcg_scores[k]):.4f}")
    print(f"Recall@{k:<3} {np.mean(recall_scores[k]):.4f}")
    print()